In [6]:
import os
import pandas as pd
import numpy as np
import re
import string
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

import lightgbm as lgb
import optuna

In [7]:
with open('../resource/recipes.pkl', 'rb') as f:
    recipes = pickle.load(f)

In [10]:
recipes["text"] = (
    recipes["Name"].fillna('') + " " +
    recipes["RecipeIngredientParts"].fillna('') + " " +
    recipes["RecipeInstructions"].fillna('')
)

recipes = recipes.dropna(subset=["AggregatedRating"]).reset_index(drop=True)

In [12]:
recipes

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions,text
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen ...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stan...","Low-Fat Berry Blue Frozen Dessert c(""blueberri..."
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_...",...,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and...","Biryani c(""saffron"", ""milk"", ""hot green chili ..."
2,40,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05T19:52:00Z,This is from one of my first Good House Keepi...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,0.0,1.8,81.5,0.4,77.2,0.3,4.0,NaN,"c(""Into a 1 quart Jar with tight fitting lid, ...","Best Lemonade c(""sugar"", ""lemons, rind of"", ""l..."
3,41,Carina's Tofu-Vegetable Kebabs,1586,Cyclopz,PT20M,PT24H,PT24H20M,1999-09-03T14:54:00Z,This dish is best prepared a day in advance to...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,0.0,1558.6,64.2,17.3,32.1,29.3,2.0,4 kebabs,"c(""Drain the tofu, carefully squeezing out exc...","Carina's Tofu-Vegetable Kebabs c(""extra firm t..."
4,42,Cabbage Soup,1538,Duckie067,PT30M,PT20M,PT50M,1999-09-19T06:19:00Z,Make and share this Cabbage Soup recipe from F...,"""https://img.sndimg.com/food/image/upload/w_55...",...,0.0,959.3,25.1,4.8,17.7,4.3,4.0,NaN,"c(""Mix everything together and bring to a boil...","Cabbage Soup c(""plain tomato juice"", ""cabbage""..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269289,540876,Backyard Breakfast Egg Salad,50509,duonyte,NaN,PT20M,PT20M,2020-08-28T20:31:00Z,I was gifted with eggs from the backyard coop ...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,287.8,511.4,6.2,0.5,1.0,13.3,4.0,NaN,"c(""1. Chop the eggs."", ""2. Combine the mayonna...","Backyard Breakfast Egg Salad c(""eggs"", ""mayonn..."
269290,540899,Butter Pecan Banana Muffins,1827868,jarp4,PT18M,PT10M,PT28M,2020-09-01T20:59:00Z,Make and share this Butter Pecan Banana Muffin...,character(0),...,17.2,253.0,28.9,1.6,15.8,3.3,12.0,12 Muffins,"c(""Heat oven to 375 degrees. Coat a standard 1...","Butter Pecan Banana Muffins c(""pecans"", ""unsal..."
269291,541030,Everything but the Bagel Seasoning (Trader Joe...,50509,duonyte,NaN,PT5M,PT5M,2020-10-02T20:48:00Z,This is a copycat of this popular seasoning mi...,character(0),...,0.0,9309.0,14.5,5.7,2.9,7.0,NaN,3/4 cup,"c(""Mix everything together and transfer to an ...",Everything but the Bagel Seasoning (Trader Joe...
269292,541195,The Most Refreshing Lemonade,2002848998,YummyFood510,PT2M,PT5M,PT7M,2020-11-09T16:13:00Z,Lemonade is so refreshing. You won't believe h...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,0.0,9.1,46.5,0.2,44.1,0.2,NaN,NaN,"c(""In a saucepan, add the sugar and 1 cup cold...","The Most Refreshing Lemonade c(""white sugar"", ..."


In [13]:
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    return text

recipes["clean_text"] = recipes["text"].apply(preprocess)

In [15]:
tfidf = TfidfVectorizer(ngram_range=(1,1), max_features=5000)
X_tfidf = tfidf.fit_transform(recipes["clean_text"])

lsa = TruncatedSVD(n_components=100, random_state=0)
X_lsa = lsa.fit_transform(X_tfidf)

count_vec = CountVectorizer(max_features=5000)
X_count = count_vec.fit_transform(recipes["clean_text"])
lda = LatentDirichletAllocation(n_components=20, random_state=0)
X_lda = lda.fit_transform(X_count)

X_final = hstack([X_tfidf, X_lsa, X_lda]).tocsr()

In [16]:
y = recipes["AggregatedRating"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

In [17]:
def objective(trial):
    param = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }
    
    gbm = lgb.LGBMRegressor(**param)
    gbm.fit(X_train, y_train, eval_set=[(X_test, y_test)])
    
    preds = gbm.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, preds))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, n_jobs=-1)

print(f"Best RMSE: {study.best_trial.value}")

[I 2026-03-29 15:46:23,017] A new study created in memory with name: no-name-53aa436d-e44b-4e35-b552-7c3da900e53d
/Users/mix/miniconda3/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-29 15:51:59,245] Trial 3 finished with value: 0.6339055374328111 and parameters: {'n_estimators': 413, 'learning_rate': 0.025072922214602104, 'num_leaves': 6, 'subsample': 0.8482456845996514, 'colsample_bytree': 0.6819608851563727, 'min_child_samples': 51}. Best is trial 3 with value: 0.6339055374328111.
/Users/mix/miniconda3/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-29 15:56:36,480] Trial 2 finished with value: 0.6328386858417578 and parameters: {'n_estimators': 285, 'learning_rate': 0.018560696760333444, 

Best RMSE: 0.6311017520339052


In [18]:
best_model = lgb.LGBMRegressor(**study.best_trial.params)
best_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.742360 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 707030
[LightGBM] [Info] Number of data points in the train set: 215435, number of used features: 4939
[LightGBM] [Info] Start training from score 4.631040


,boosting_type,'gbdt'
,num_leaves,135
,max_depth,-1
,learning_rate,0.031264602958054
,n_estimators,206
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,76


In [20]:
with open('../resource/recipe_recommendation_model.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'tfidf': tfidf,
        'lsa': lsa,
        'count_vec': count_vec,
        'lda': lda,
        'X_final': X_final 
    }, f)

In [21]:
def recommend_for_user(user_recipe_ids, top_k=10):
    indices = recipes[recipes["RecipeId"].isin(user_recipe_ids)].index
    
    if len(indices) == 0:
        return "No valid user recipes"
        
    user_profile = X_final[indices].mean(axis=0)
    similarity_scores = np.asarray(X_final @ user_profile.T).flatten()
    
    predicted_ratings = best_model.predict(X_final)
    
    scaler = MinMaxScaler()
    normalized_similarity = scaler.fit_transform(similarity_scores.reshape(-1, 1)).flatten()
    normalized_ratings = scaler.fit_transform(predicted_ratings.reshape(-1, 1)).flatten()
    
    recipes["final_score"] = (normalized_similarity * 0.7) + (normalized_ratings * 0.3)
    
    recs = recipes.drop(indices).sort_values("final_score", ascending=False)
    return recs[["RecipeId", "Name", "AggregatedRating", "final_score"]].head(top_k)

In [22]:
sample_recipes = recipes.sample(5)
sample_ids = sample_recipes["RecipeId"].tolist()

display(sample_recipes[["RecipeId", "Name"]])
display(recommend_for_user(sample_ids))

,RecipeId,Name
225078,380077,Everybody Loves 'em Peanut Butter Cookies
251260,454323,Chinese Rice Cakes
168902,263410,River Road Cookbook Sweet and Sour Pork
30536,43215,Easy Blueberry Cream Pie
123938,182915,Spinach &amp; Pear Salad With Dijon Mustard Vi...


/Users/mix/miniconda3/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,RecipeId,Name,AggregatedRating,final_score
189042,302017,Peanut Butter Cup Cookies,5.0,0.915167
209899,346008,Peanut Butter Cookies - the Magnolia Bakery,5.0,0.898484
82549,115689,Evil Peanut Butter Cookies,4.0,0.898106
238771,415283,Momofuku Crack Pie,5.0,0.892144
238720,415161,Chocolate Peanut Butter Cupcakes,3.0,0.889630
257813,473941,Chewy Peanut Butter Chocolate Chip Cookies,5.0,0.882141
92556,131534,Outrageous Chocolate Chip Cookies,5.0,0.880913
50994,70747,Butterfinger Crumb Cake,4.0,0.879199
25662,36732,Peaches and Cream Jelly Roll,5.0,0.877347
158934,244642,Blackberry Cream Cheese Coffee Cake,4.5,0.872200


In [23]:
with open('../resource/recipe_recommendation_model.pkl', 'rb') as f:
    model_data = pickle.load(f)

In [24]:
with open('../resource/recipes.pkl', 'rb') as f:
    recipes_full = pickle.load(f)

In [25]:
recipes_full = recipes_full.copy()

In [ ]:
recipes["text"] = (
    recipes["Name"].fillna('') + " " +
    recipes["RecipeIngredientParts"].fillna('') + " " +
    recipes["RecipeInstructions"].fillna('')
)